In [6]:
import numpy as np
import pandas as pd
import os

os.getcwd()
os.chdir("CSV-Data")

In [ ]:
gpa   = pd.read_csv("avg_gpa_by_school.csv")
nclex = pd.read_csv("GA_RN_NCLEX_Pass_Rates.csv")

# Clean column names 
gpa.columns   = gpa.columns.str.strip()
nclex.columns = nclex.columns.str.strip()

# Standardize school names 
gpa['Schools_Cleaned']   = gpa['School'].str.strip().str.lower()
nclex['Schools_Cleaned'] = nclex['Registered Nursing Program Name'].str.strip().str.lower()

#  1. GPA (already aggregated per school) 
gpa_df = gpa[['Schools_Cleaned', 'Average GPA']]

#  2. NCLEX Pass Rate 
nclex_df = nclex[['Schools_Cleaned', '4 Year Average']].rename(
    columns={'4 Year Average': 'NCLEX Pass Rate'}
)

# Convert to numeric in case of stray % signs or spaces
nclex_df['NCLEX Pass Rate'] = pd.to_numeric(
    nclex_df['NCLEX Pass Rate'].astype(str).str.replace('%', '', regex=False),
    errors='coerce'
)

#  Merge 
df = gpa_df.merge(nclex_df, on='Schools_Cleaned', how='inner')

#  Z-score function 
def zscore(col):
    return (col - col.mean()) / col.std()

df['GPA z-score']             = zscore(df['Average GPA'])
df['NCLEX Pass Rate z-score'] = zscore(df['NCLEX Pass Rate'])

#  Final weighted score 
df['Final Score'] = (
    df['GPA z-score']             * 0.60 +
    df['NCLEX Pass Rate z-score'] * 0.40
)

#  Sort best to worst 
df = df.sort_values('Final Score', ascending=False).reset_index(drop=True)

#  Save 
df.to_csv('school_scores.csv', index=False)

print(df[['Schools_Cleaned', 'Average GPA', 'NCLEX Pass Rate', 'Final Score']].round(3))

                         Schools_Cleaned  Average GPA  NCLEX Pass Rate  \
0                   reinhardt university        3.650            94.74   
1               georgia state university        3.571            97.75   
2            university of north georgia        3.591            93.05   
3                       emory university        3.624            90.48   
4              kennesaw state university        3.572            91.28   
5               georgia gwinnett college        3.507            93.36   
6                       emory university        3.624            85.76   
7                      brenau university        3.435            94.55   
8               georgia state university        3.571            87.65   
9             university of west georgia        3.585            83.33   
10             columbus state university        3.317            94.55   
11                     brenau university        3.435            88.41   
12  georgia college and state universi